# NB01 — Combined Cross-Dataset Exploration

**Design:** each dataset contributes both members (used to train the authenticator)
and non-members (used as MIA test subjects).

- **whuGAIT D1** IDs 21–118 → 98 members (same as the whuGAIT run)
- **UCI-HAR** train-split subjects → 21 members (IDs 1001+)
  test-split subjects  → 9 non-members (IDs 1001+)
- **WISDM** first 40 subjects (raw 1600–1639) → 40 members (IDs 2001–2040)
  last 11 subjects  (raw 1640–1650) → 11 non-members (IDs 2041–2051)

**Observation:** the UCI_HAR ids have been mapped to 1000 for an easier analisis,
the same logic is applied to WISDM ids.

Training pairs are built within each dataset separately and then concatenated,
so the LSTM never sees cross-dataset different-person pairs that the CNN could
trivially separate by sensor statistics.

Test pairs are built from UCI-HAR and WISDM non-members only.

**Outputs:**
- `artifacts/combined/subject_split.json`
- `artifacts/combined/auth_pairs.npz`
- `latex/generated/nb01_combined_metrics.tex`


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import sys
sys.path.insert(0, '..')

import json
import logging
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.data.dataset         import load_dataset, filter_subjects
from src.data.ucihar_dataset  import load_ucihar
from src.data.wisdm_dataset   import load_wisdm
from src.data.pair_builder    import build_auth_pairs
from src.utils.config_loader  import load_config
from src.utils.latex_writer   import write_latex_metrics

DATASET  = 'combined'  # <<< RUNNER INJECTS THIS

DATA_ROOT    = Path('../data')
LOG_DIR      = Path('../logs')      / DATASET
ARTIFACT_DIR = Path('../artifacts') / DATASET
RESULT_DIR   = Path('../results')   / DATASET
for _d in [LOG_DIR, ARTIFACT_DIR, RESULT_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

WHUGAIT_ROOT = DATA_ROOT / 'Dataset #1'
UCI_ROOT     = DATA_ROOT / 'UCI HAR Dataset'
WISDM_ROOT   = DATA_ROOT / 'WISDM'

# ── Load combined config ──
cfg  = load_config()['combined']
_rng = np.random.default_rng(cfg['subject_seed'])
WHU_MEMBER_IDS    = sorted(_rng.choice(range(21, 119), size=cfg['whuGAIT_n_members'],    replace=False).tolist())
WHU_NONMEMBER_IDS = sorted(_rng.choice(range(1,  21),  size=cfg['whuGAIT_n_nonmembers'], replace=False).tolist())

# ── ID spaces (no overlap) ──
UCI_ID_OFFSET = 1000   # raw UCI 1–30  → 1001–1030
WIS_ID_OFFSET = 401    # raw WISDM 1600–1650 → 2001–2051
N_WIS_MEMBERS = 40     # first 40 WISDM raw IDs → members

# ── pairs per class ──
N_TRAIN_WHU  = cfg['n_train_pairs_per_class']
N_TRAIN_UCI  = cfg['n_train_pairs_per_class']
N_TRAIN_WIS  = cfg['n_train_pairs_per_class']
N_TEST_PAIRS = cfg['n_test_pairs_per_class']

logging.basicConfig(level=logging.INFO, format='%(message)s')
log = logging.getLogger('nb01_combined')
log.info('=== NB01 — Combined Cross-Dataset Exploration ===')
log.info(f'whuGAIT members ({len(WHU_MEMBER_IDS)}):     {WHU_MEMBER_IDS}')
log.info(f'whuGAIT non-members ({len(WHU_NONMEMBER_IDS)}): {WHU_NONMEMBER_IDS}')

## 1. whuGAIT — Member windows (IDs 21–118)

In [ ]:
X_whu_tr, y_whu_tr = load_dataset(str(WHUGAIT_ROOT), 'train')
X_whu_te, y_whu_te = load_dataset(str(WHUGAIT_ROOT), 'test')

# Member windows
X_whu_mem_tr, y_whu_mem_tr = filter_subjects(X_whu_tr, y_whu_tr, WHU_MEMBER_IDS)
X_whu_mem_te, y_whu_mem_te = filter_subjects(X_whu_te, y_whu_te, WHU_MEMBER_IDS)
X_whu = np.concatenate([X_whu_mem_tr, X_whu_mem_te], axis=0)
y_whu = np.concatenate([y_whu_mem_tr, y_whu_mem_te], axis=0)

# Non-member windows
X_whu_nm_tr, y_whu_nm_tr = filter_subjects(X_whu_tr, y_whu_tr, WHU_NONMEMBER_IDS)
X_whu_nm_te, y_whu_nm_te = filter_subjects(X_whu_te, y_whu_te, WHU_NONMEMBER_IDS)
X_whu_nm = np.concatenate([X_whu_nm_tr, X_whu_nm_te], axis=0)
y_whu_nm = np.concatenate([y_whu_nm_tr, y_whu_nm_te], axis=0)

assert set(np.unique(y_whu).tolist()) == set(WHU_MEMBER_IDS), 'Missing whuGAIT members'
assert set(np.unique(y_whu_nm).tolist()) == set(WHU_NONMEMBER_IDS), 'Missing whuGAIT non-members'
log.info(f'whuGAIT members:     {len(WHU_MEMBER_IDS)} subjects  |  {len(y_whu)} windows')
log.info(f'whuGAIT non-members: {len(WHU_NONMEMBER_IDS)} subjects  |  {len(y_whu_nm)} windows')

## 2. UCI-HAR — train subjects as members, test subjects as non-members

UCI-HAR's original train/test split naturally separates subjects (not windows),
so we use it directly as our member/non-member boundary.


In [ ]:
# UCI-HAR train split → members (21 subjects)
X_uci_mem, y_uci_mem_raw = load_ucihar(str(UCI_ROOT), 'train', gait_only=True)
y_uci_mem = y_uci_mem_raw + UCI_ID_OFFSET

# UCI-HAR test split → non-members (9 subjects)
X_uci_nm, y_uci_nm_raw = load_ucihar(str(UCI_ROOT), 'test', gait_only=True)
y_uci_nm = y_uci_nm_raw + UCI_ID_OFFSET

uci_member_ids    = sorted(np.unique(y_uci_mem).tolist())
uci_nonmember_ids = sorted(np.unique(y_uci_nm).tolist())

# Sanity: no overlap
assert len(set(uci_member_ids) & set(uci_nonmember_ids)) == 0
assert len(set(uci_member_ids) & set(WHU_MEMBER_IDS)) == 0

log.info(f'UCI-HAR members:     {len(uci_member_ids)} subjects  |  {len(y_uci_mem)} windows')
log.info(f'  IDs: {uci_member_ids}')
log.info(f'UCI-HAR non-members: {len(uci_nonmember_ids)} subjects  |  {len(y_uci_nm)} windows')
log.info(f'  IDs: {uci_nonmember_ids}')
print(f'UCI-HAR  members={len(uci_member_ids)}  non-members={len(uci_nonmember_ids)}')


## 3. WISDM — first 40 subjects as members, remaining 11 as non-members

Raw WISDM subject IDs are 1600–1650 (51 subjects). We remap them to 2001–2051
to avoid collisions with whuGAIT (1–118) and UCI-HAR (1001–1030).


In [ ]:
X_wis_all, y_wis_raw = load_wisdm(str(WISDM_ROOT), gait_only=True, window=128, stride=64)
y_wis_all = y_wis_raw + WIS_ID_OFFSET   # 1600–1650 → 2001–2051

wis_all_ids = sorted(np.unique(y_wis_all).tolist())
wis_member_ids    = wis_all_ids[:N_WIS_MEMBERS]    # first 40
wis_nonmember_ids = wis_all_ids[N_WIS_MEMBERS:]    # last 11

mem_mask = np.isin(y_wis_all, wis_member_ids)
X_wis_mem = X_wis_all[mem_mask];  y_wis_mem = y_wis_all[mem_mask]
X_wis_nm  = X_wis_all[~mem_mask]; y_wis_nm  = y_wis_all[~mem_mask]

assert len(set(wis_member_ids) & set(wis_nonmember_ids)) == 0
assert len(set(wis_member_ids) & set(WHU_MEMBER_IDS)) == 0
assert len(set(wis_member_ids) & set(uci_member_ids)) == 0

log.info(f'WISDM members:     {len(wis_member_ids)} subjects  |  {len(y_wis_mem)} windows  IDs {min(wis_member_ids)}–{max(wis_member_ids)}')
log.info(f'WISDM non-members: {len(wis_nonmember_ids)} subjects  |  {len(y_wis_nm)} windows  IDs {min(wis_nonmember_ids)}–{max(wis_nonmember_ids)}')
print(f'WISDM  members={len(wis_member_ids)}  non-members={len(wis_nonmember_ids)}')


## 4. Subject split

In [ ]:
all_member_ids    = WHU_MEMBER_IDS + uci_member_ids + wis_member_ids
all_nonmember_ids = WHU_NONMEMBER_IDS + uci_nonmember_ids + wis_nonmember_ids

assert len(set(all_member_ids) & set(all_nonmember_ids)) == 0, 'Member/non-member overlap!'

split = {
    'dataset':             'combined',
    'train_ids':           all_member_ids,
    'held_out_ids':        all_nonmember_ids,
    'n_train':             len(all_member_ids),
    'n_held_out':          len(all_nonmember_ids),
    'member_whuGAIT':      WHU_MEMBER_IDS,
    'member_ucihar':       uci_member_ids,
    'member_wisdm':        wis_member_ids,
    'held_out_whuGAIT':    WHU_NONMEMBER_IDS,
    'held_out_ucihar':     uci_nonmember_ids,
    'held_out_wisdm':      wis_nonmember_ids,
    'id_offset_ucihar':    UCI_ID_OFFSET,
    'id_offset_wisdm':     WIS_ID_OFFSET,
}
with open(ARTIFACT_DIR / 'subject_split.json', 'w') as f:
    json.dump(split, f, indent=2)

log.info(f'Members:     {len(all_member_ids)}  (whuGAIT={len(WHU_MEMBER_IDS)} + UCI={len(uci_member_ids)} + WISDM={len(wis_member_ids)})')
log.info(f'Non-members: {len(all_nonmember_ids)}  (whuGAIT={len(WHU_NONMEMBER_IDS)} + UCI={len(uci_nonmember_ids)} + WISDM={len(wis_nonmember_ids)})')
log.info('subject_split.json saved.')
print(f'Members: {len(all_member_ids)}   Non-members: {len(all_nonmember_ids)}')

## 5. Channel statistics per dataset

In [ ]:
CHANNEL_NAMES = ['acc_x', 'acc_y', 'acc_z', 'gyr_x', 'gyr_y', 'gyr_z']

def _ch_stats(X, name):
    mean = X.mean(axis=(0, 2))
    std  = X.std(axis=(0, 2))
    log.info(f'{name}:')
    for ch, m, s in zip(CHANNEL_NAMES, mean, std):
        log.info(f'  {ch:8s}  mean={m:+.4f}  std={s:.4f}')
    return mean, std

whu_mean, whu_std = _ch_stats(X_whu,     'whuGAIT members')
uci_mean, uci_std = _ch_stats(X_uci_mem, 'UCI-HAR members')
wis_mean, wis_std = _ch_stats(X_wis_mem, 'WISDM members')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, mean, std, color) in zip(axes, [
    ('whuGAIT members',   whu_mean, whu_std, '#3498db'),
    ('UCI-HAR members',   uci_mean, uci_std, '#2ecc71'),
    ('WISDM members',     wis_mean, wis_std, '#e74c3c'),
]):
    x = range(6)
    ax.bar(x, mean, yerr=std, color=color, alpha=0.7, capsize=4)
    ax.set_xticks(x); ax.set_xticklabels(CHANNEL_NAMES, rotation=45)
    ax.set_title(name)
    ax.set_ylabel('Mean ± std')
plt.tight_layout()
plt.savefig(RESULT_DIR / '01_combined_channel_stats.png', dpi=120)
plt.show()
log.info('Figure saved: 01_combined_channel_stats.png')


## 6. Auth training pairs — all members

Pairs are built **within each dataset separately** then concatenated.
This prevents the model from learning cross-dataset sensor statistics as a
proxy for identity, which would make the later MIA uninterpretable.


In [ ]:
# whuGAIT member pairs
X1_whu, X2_whu, y_whu_p, s1_whu, s2_whu = build_auth_pairs(
    X_whu, y_whu, n_pairs_per_class=N_TRAIN_WHU, seed=42)

# UCI-HAR member pairs
X1_uci_m, X2_uci_m, y_uci_m_p, s1_uci_m, s2_uci_m = build_auth_pairs(
    X_uci_mem, y_uci_mem, n_pairs_per_class=N_TRAIN_UCI, seed=43)

# WISDM member pairs
X1_wis_m, X2_wis_m, y_wis_m_p, s1_wis_m, s2_wis_m = build_auth_pairs(
    X_wis_mem, y_wis_mem, n_pairs_per_class=N_TRAIN_WIS, seed=44)

# Concatenate all training pairs
X1_tr    = np.concatenate([X1_whu,  X1_uci_m,  X1_wis_m],  axis=0)
X2_tr    = np.concatenate([X2_whu,  X2_uci_m,  X2_wis_m],  axis=0)
y_tr     = np.concatenate([y_whu_p, y_uci_m_p, y_wis_m_p], axis=0)
subj1_tr = np.concatenate([s1_whu,  s1_uci_m,  s1_wis_m],  axis=0)
subj2_tr = np.concatenate([s2_whu,  s2_uci_m,  s2_wis_m],  axis=0)

# Shuffle the full training set
rng_shuffle = np.random.default_rng(0)
perm = rng_shuffle.permutation(len(y_tr))
X1_tr, X2_tr, y_tr = X1_tr[perm], X2_tr[perm], y_tr[perm]
subj1_tr, subj2_tr = subj1_tr[perm], subj2_tr[perm]

member_set = set(all_member_ids)
assert set(subj1_tr.tolist() + subj2_tr.tolist()) <= member_set, 'Non-member in training pairs!'

log.info(f'Train pairs total:  {len(y_tr)}  (same={int((y_tr==0).sum())}  diff={int((y_tr==1).sum())})')
log.info(f'  whuGAIT:  {len(y_whu_p)}  UCI-HAR: {len(y_uci_m_p)}  WISDM: {len(y_wis_m_p)}')
print(f'Training pairs: {len(y_tr)} total  (whuGAIT={len(y_whu_p)}, UCI={len(y_uci_m_p)}, WISDM={len(y_wis_m_p)})')


## 7. Auth test pairs — non-members (UCI-HAR + WISDM)

Built within each dataset separately to keep per-dataset analysis clean.
Subject IDs in subj1_te/subj2_te allow attribution in NB04/05.


In [ ]:
# whuGAIT non-member test pairs
X1_whu_nm, X2_whu_nm, y_whu_nm_p, s1_whu_nm, s2_whu_nm = build_auth_pairs(
    X_whu_nm, y_whu_nm, n_pairs_per_class=N_TEST_PAIRS, seed=52)

# UCI-HAR non-member test pairs
X1_uci_nm, X2_uci_nm, y_uci_nm_p, s1_uci_nm, s2_uci_nm = build_auth_pairs(
    X_uci_nm, y_uci_nm, n_pairs_per_class=N_TEST_PAIRS, seed=50)

# WISDM non-member test pairs
X1_wis_nm, X2_wis_nm, y_wis_nm_p, s1_wis_nm, s2_wis_nm = build_auth_pairs(
    X_wis_nm, y_wis_nm, n_pairs_per_class=N_TEST_PAIRS, seed=51)

X1_te    = np.concatenate([X1_whu_nm,  X1_uci_nm,  X1_wis_nm],  axis=0)
X2_te    = np.concatenate([X2_whu_nm,  X2_uci_nm,  X2_wis_nm],  axis=0)
y_te     = np.concatenate([y_whu_nm_p, y_uci_nm_p, y_wis_nm_p], axis=0)
subj1_te = np.concatenate([s1_whu_nm,  s1_uci_nm,  s1_wis_nm],  axis=0)
subj2_te = np.concatenate([s2_whu_nm,  s2_uci_nm,  s2_wis_nm],  axis=0)

nonmember_set = set(all_nonmember_ids)
assert set(subj1_te.tolist() + subj2_te.tolist()) <= nonmember_set, 'Member in test pairs!'

log.info(f'Test pairs: whuGAIT={len(y_whu_nm_p)}  UCI-HAR={len(y_uci_nm_p)}  WISDM={len(y_wis_nm_p)}  total={len(y_te)}')
log.info(f'  same={int((y_te==0).sum())}  diff={int((y_te==1).sum())}')
print(f'Test pairs: {len(y_te)} total  (whuGAIT={len(y_whu_nm_p)}, UCI={len(y_uci_nm_p)}, WISDM={len(y_wis_nm_p)})')

## 8. Save

In [ ]:
np.savez_compressed(
    ARTIFACT_DIR / 'auth_pairs.npz',
    X1_tr=X1_tr,    X2_tr=X2_tr,    y_tr=y_tr,
    subj1_tr=subj1_tr, subj2_tr=subj2_tr,
    X1_te=X1_te,    X2_te=X2_te,    y_te=y_te,
    subj1_te=subj1_te, subj2_te=subj2_te,
)
log.info(f'Saved: {ARTIFACT_DIR}/auth_pairs.npz')

summary = f"""
=== NB01 COMBINED SUMMARY ===

Members ({len(all_member_ids)} total):
  whuGAIT:  {len(WHU_MEMBER_IDS)} subjects (IDs 21–118)    {len(X_whu)} windows
  UCI-HAR:  {len(uci_member_ids)} subjects (IDs {min(uci_member_ids)}–{max(uci_member_ids)})  {len(X_uci_mem)} windows
  WISDM:    {len(wis_member_ids)} subjects (IDs {min(wis_member_ids)}–{max(wis_member_ids)})  {len(X_wis_mem)} windows

Non-members ({len(all_nonmember_ids)} total):
  UCI-HAR:  {len(uci_nonmember_ids)} subjects (IDs {min(uci_nonmember_ids)}–{max(uci_nonmember_ids)})  {len(X_uci_nm)} windows
  WISDM:    {len(wis_nonmember_ids)} subjects (IDs {min(wis_nonmember_ids)}–{max(wis_nonmember_ids)})  {len(X_wis_nm)} windows

Training pairs: {len(y_tr)} (same={int((y_tr==0).sum())}  diff={int((y_tr==1).sum())})
  whuGAIT:  {len(y_whu_p)}
  UCI-HAR:  {len(y_uci_m_p)}
  WISDM:    {len(y_wis_m_p)}

Test pairs: {len(y_te)} (same={int((y_te==0).sum())}  diff={int((y_te==1).sum())})
  UCI-HAR:  {len(y_uci_nm_p)}
  WISDM:    {len(y_wis_nm_p)}

Saved: artifacts/combined/subject_split.json
       artifacts/combined/auth_pairs.npz
"""
print(summary)
log.info(summary)


In [ ]:
write_latex_metrics('nb01_combined', {
    'combinedTotalMembers':        len(all_member_ids),
    'combinedMemberWhu':           len(WHU_MEMBER_IDS),
    'combinedMemberUci':           len(uci_member_ids),
    'combinedMemberWis':           len(wis_member_ids),
    'combinedTotalNonmembers':     len(all_nonmember_ids),
    'combinedNonmemberUci':        len(uci_nonmember_ids),
    'combinedNonmemberWis':        len(wis_nonmember_ids),
    'combinedTrainPairs':          len(y_tr),
    'combinedTestPairs':           len(y_te),
}, output_dir=Path('../latex/generated'))
log.info('LaTeX metrics written.')
